# **Part 4: Recycling Instruction Generation with RAG**

Given a waste category, the system retrieves the matching policy text from the waste policy documents. A fine-tuned small language model, `flan-t5-small`, then uses the retrieved policy information to generate recycling instructions. A checker verifies the generated instructions against the retrieved policy text to help ensure that the response is consistent with the available information.

This section requires `waste_descriptions.csv` and `waste_policy_documents.json` to be available alongside the notebook. It also uses PyTorch, Transformers, Sentence Transformers, and TextStat. The fine-tuned model is saved in the `waste_rag_model` folder, which is loaded and used in Part 5.

> **Note:** Fine-tuning the model takes approximately 20 minutes when running on CPU. The unseen-category evaluation took approximately 17 minutes in the recorded run.


## **Step 1: Loading Libraries and Checking the Machine**

In [4]:
import os
os.environ["USE_TF"] = "0"          # keep transformers from loading TensorFlow
import re
import json
import time
import copy
import random
import collections
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("torch", torch.__version__)

torch 2.11.0+cpu


## **Step 2: Policy Chunks**

In this step, each policy document is divided into smaller sections, or *chunks*, to make information retrieval more efficient. Each chunk is tagged with its corresponding waste category, allowing the system to quickly locate relevant policy information during the RAG process. These categorized chunks form the knowledge base that the language model uses to generate accurate recycling instructions.


In [5]:
# Load the waste descriptions dataset
desc = pd.read_csv("waste_descriptions.csv")

# Load the waste policy documents
policies = json.load(open("waste_policy_documents.json"))

# Create a pattern for identifying section headers in the policy documents
header_pattern = re.compile(r"^([A-Za-z][A-Za-z \-/&]+):\s*$", re.M)

# Function to split each policy document into sections
def split_sections(text):

    # Split the document using the section header pattern
    parts = header_pattern.split(text)

    # Return the section headers together with their corresponding text
    return list(zip(parts[1::2], [b.strip() for b in parts[2::2]]))

# Create an empty list to store the policy chunks
chunks = []

# Loop through each policy document
for policy in policies:

    # Split the current policy document into sections
    for header, body in split_sections(policy["document_text"]):

        # Get the waste categories covered by the policy
        categories = policy["categories_covered"]

        # Check whether the section header identifies a specific category
        match = re.match(r"([A-Z ]+) GUIDELINES$", header)

        # Handle policies that cover more than one category
        if len(categories) > 1:

            # Mark general requirements as applying to all categories
            if header == "GENERAL REQUIREMENTS":
                categories = ["ALL"]

            # Match a section with its specific waste category
            elif match:
                own = [c for c in categories if c.upper() == match.group(1).strip()]
                categories = own or categories

        # Add the policy section as a chunk
        chunks.append({"chunk_id": f"policy{policy['policy_id']}::{header}", "policy_id": policy["policy_id"],
                       "header": header, "category": categories[0],
                       "text": f"{policy['policy_type']} | {header}\n{body}"})

# Convert the policy chunks into a DataFrame
chunk_table = pd.DataFrame(chunks)

# Display the number of chunks created before filtering
print("chunks before filtering:", len(chunk_table))

chunks before filtering: 67


In [6]:
# Keep the category documents (1-9) and one copy of the general rules
own_docs = chunk_table[chunk_table["policy_id"] <= 9]
general_chunk = chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"].head(1)

# Create one disposal card for each waste category using the CSV data
cards = []

# Group the waste descriptions by category
for category, group in desc.groupby("category"):

    # Get the unique disposal instructions for the category
    instructions = sorted(group["disposal_instruction"].dropna().unique())

    # Get the common confusion notes for the category
    note = group["common_confusion"].dropna().unique()

    # Create the text for the disposal card
    text = (f"{category} | Disposal card\nComposition: {group['material_composition'].iloc[0]}\n"
            f"Disposal: " + " ".join(instructions))

    # Add the common confusion note if one is available
    if len(note) > 0:
        text += "\nCommon confusion: " + note[0]

    # Add the disposal card to the list
    cards.append({"chunk_id": f"card::{category}", "policy_id": None, "header": "Disposal card",
                  "category": category, "text": text})

# Combine the policy chunks, general rules, and disposal cards into one RAG corpus
rag_corpus = pd.concat([own_docs, general_chunk, pd.DataFrame(cards)], ignore_index=True)

# Get the list of unique waste categories
CATEGORIES = sorted(desc["category"].unique())

# Display the total number of chunks
print("chunks:", len(rag_corpus))

# Display the number of chunks for each category
print(rag_corpus["category"].value_counts().to_dict())

# Display summary statistics for the length of the chunks
print(rag_corpus["text"].str.split().str.len().describe().round(0).to_dict())

chunks: 54
{'Textile Trash': 6, 'Glass': 6, 'Food Organics': 6, 'Plastic': 6, 'Vegetation': 6, 'Cardboard': 6, 'Metal': 6, 'Paper': 6, 'Miscellaneous Trash': 5, 'ALL': 1}
{'count': 54.0, 'mean': 31.0, 'std': 18.0, 'min': 14.0, '25%': 18.0, '50%': 26.0, '75%': 31.0, 'max': 75.0}


The final RAG corpus contains **54 chunks** covering all 9 waste categories, plus one general requirements chunk. Most categories have 6 chunks, while Miscellaneous Trash has 5.

The chunks contain an average of approximately **31 words**, with lengths ranging from **14 to 75 words**. This provides relatively small sections of policy information for the retrieval step.

## **Step 3: Embeddings and the Retrieval Index**

Two retrieval indexes are created over the 54 policy chunks. The first uses **TF-IDF**, which matches words and terms between the query and the policy text. The second uses **dense MiniLM embeddings**, which represent each chunk as a vector of **384 numbers** and allow the system to find text with similar meaning.

Since the corpus contains only 54 chunks, each index can be represented as a matrix, with **cosine similarity** used to find the chunks most relevant to a given query.


In [7]:
# Convert the policy chunk text into a list
chunk_texts = rag_corpus["text"].tolist()

# Store the category for each policy chunk
chunk_category = rag_corpus["category"].values

# Create the TF-IDF vectorizer
# Uses single words and pairs of words (bigrams)
tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")

# Convert the policy chunks into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(chunk_texts)

# Load the MiniLM sentence embedding model
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Convert each policy chunk into a dimensional dense embedding
dense_matrix = encoder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False)

# Display the shape of the TF-IDF index
print("TF-IDF index:", tfidf_matrix.shape)

# Display the shape of the dense embedding index
print("dense index:", dense_matrix.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TF-IDF index: (54, 1170)
dense index: (54, 384)


In [8]:
# Calculate TF-IDF similarity scores for a question
def tfidf_scores(question):

    # Compare the question with all policy chunks using cosine similarity
    return cosine_similarity(tfidf.transform([question]), tfidf_matrix)[0]


# Calculate dense embedding similarity scores for a question
def dense_scores(question):

    # Compare the question embedding with all policy chunk embeddings
    return dense_matrix @ encoder.encode([question], normalize_embeddings=True)[0]


# Combine TF-IDF and dense embedding scores
def hybrid_scores(question, alpha=0.5):

    # Get the scores from both retrieval methods
    a, b = tfidf_scores(question), dense_scores(question)

    # Standardize the TF-IDF scores
    a = (a - a.mean()) / (a.std() + 1e-9)

    # Standardize the dense embedding scores
    b = (b - b.mean()) / (b.std() + 1e-9)

    # Combine both scores using the selected TF-IDF weight
    return alpha * a + (1 - alpha) * b          # alpha is the weight of TF-IDF

## **Step 4: Compare the Retrieval Methods**

In this step, the different retrieval methods are evaluated to see how well they identify relevant policy chunks. A chunk will be considered relevant if it belongs to the correct waste category for the question.

The evaluation uses **54 questions in total**: 36 category-based questions, consisting of 4 different wordings for each of the 9 waste categories, and 18 item-based questions that do not explicitly include the waste category name. The item questions, such as *“How do I recycle a pizza box?”*, were written manually to test whether the retrieval system can identify the correct category from the item description alone.


In [9]:
wordings = ["How to recycle {c}", "How do I dispose of {c} properly?",
            "What is the correct way to recycle {c}?", "{c} recycling rules"]
category_questions = [(w.format(c=c), c) for c in CATEGORIES for w in wordings]

item_questions = [
    ("How do I recycle a pizza box?", "Cardboard"), ("What should I do with shipping boxes?", "Cardboard"),
    ("How do I dispose of banana peels?", "Food Organics"), ("Can I compost coffee grounds and eggshells?", "Food Organics"),
    ("How do I recycle a wine bottle?", "Glass"), ("What do I do with a broken glass jar?", "Glass"),
    ("How do I recycle aluminium cans?", "Metal"), ("Can I recycle an empty aerosol can?", "Metal"),
    ("How do I dispose of old batteries?", "Miscellaneous Trash"), ("How do I dispose of light bulbs?", "Miscellaneous Trash"),
    ("How do I recycle old newspapers?", "Paper"), ("What do I do with shredded documents?", "Paper"),
    ("How do I recycle a plastic bottle?", "Plastic"), ("Can I recycle plastic bags?", "Plastic"),
    ("What should I do with old clothes?", "Textile Trash"), ("How do I dispose of worn towels?", "Textile Trash"),
    ("What do I do with grass clippings?", "Vegetation"), ("How do I dispose of tree branches?", "Vegetation"),
]

In [10]:
# Function to evaluate how well a retrieval method finds the correct category
def evaluate_retrieval(scores_function, questions):

    # Create lists to store the evaluation results
    hit1, hit3, hit5, reciprocal = [], [], [], []

    # Evaluate each question
    for question, category in questions:

        # Rank all chunks from highest to lowest similarity score
        order = np.argsort(-scores_function(question))

        # Check which retrieved chunks belong to the correct category
        relevant = chunk_category[order] == category

        # Check whether a relevant chunk appears in the top 1 result
        hit1.append(relevant[:1].any())

        # Check whether a relevant chunk appears in the top 3 results
        hit3.append(relevant[:3].any())

        # Check whether a relevant chunk appears in the top 5 results
        hit5.append(relevant[:5].any())

        # Find the position of the first relevant chunk
        first = np.where(relevant)[0]

        # Calculate the reciprocal rank of the first relevant result
        reciprocal.append(1 / (first[0] + 1) if len(first) else 0)

    # Return the retrieval evaluation metrics
    return {"MRR": round(float(np.mean(reciprocal)), 3), "hit@1": round(float(np.mean(hit1)), 3),
            "hit@3": round(float(np.mean(hit3)), 3), "hit@5": round(float(np.mean(hit5)), 3)}


# Create an empty list to store results from all retrieval methods
rows = []

# Define the retrieval methods being compared
methods = {"TF-IDF": tfidf_scores, "dense (MiniLM)": dense_scores, "hybrid 0.5": hybrid_scores}

# Evaluate each retrieval method on both question sets
for method_name, scores_function in methods.items():

    # Evaluate category questions and item questions separately
    for set_name, questions in [("category questions", category_questions), ("item questions", item_questions)]:

        # Store the evaluation results
        rows.append({"method": method_name, "questions": set_name, **evaluate_retrieval(scores_function, questions)})

# Display the evaluation results
print(pd.DataFrame(rows).to_string(index=False))

        method          questions   MRR  hit@1  hit@3  hit@5
        TF-IDF category questions 0.986  0.972  1.000  1.000
        TF-IDF     item questions 0.783  0.667  0.944  0.944
dense (MiniLM) category questions 0.940  0.917  0.944  1.000
dense (MiniLM)     item questions 0.944  0.889  1.000  1.000
    hybrid 0.5 category questions 0.986  0.972  1.000  1.000
    hybrid 0.5     item questions 0.917  0.833  1.000  1.000


TF-IDF performs best on the category questions, achieving an MRR of 0.986 and 97.2% Hit@1. This is expected because these questions explicitly contain the waste category name.

For the item questions, dense MiniLM performs best, with an MRR of 0.944 and 88.9% Hit@1. This shows that semantic embeddings are particularly useful when the question describes an item without directly naming its waste category.

The hybrid method combines both approaches and achieves 100% Hit@3 and Hit@5 on both question sets, showing that combining word-based and semantic retrieval provides reliable top-3 and top-5 retrieval.

## **Step 5: Choose the Hybrid Weight and Look at the Failures**

In this step, different values of the hybrid retrieval weight are tested to determine how much importance should be given to TF-IDF compared with dense MiniLM embeddings. The results are compared to select a suitable hybrid weight for the RAG system.

The retrieval failures are also examined to understand which questions are difficult for the system and where the retrieval process could be improved.


In [11]:
# Create an empty list to store the results for each hybrid weight
rows = []

# Test different weights for TF-IDF
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:

    # Create a retrieval function using the current TF-IDF weight
    scores_function = lambda question, a=alpha: hybrid_scores(question, a)

    # Evaluate the current weight using category questions
    cat_result = evaluate_retrieval(scores_function, category_questions)

    # Evaluate the current weight using item questions
    item_result = evaluate_retrieval(scores_function, item_questions)

    # Store the results for the current weight
    rows.append({"alpha (TF-IDF weight)": alpha, "category MRR": cat_result["MRR"],
                 "category hit@1": cat_result["hit@1"], "item MRR": item_result["MRR"], "item hit@1": item_result["hit@1"]})

# Display the results for all tested weights
print(pd.DataFrame(rows).to_string(index=False))

 alpha (TF-IDF weight)  category MRR  category hit@1  item MRR  item hit@1
                  0.00         0.940           0.917     0.944       0.889
                  0.25         1.000           1.000     0.944       0.889
                  0.50         0.986           0.972     0.917       0.833
                  0.75         0.986           0.972     0.843       0.722
                  1.00         0.986           0.972     0.783       0.667


In [12]:
# Function to display questions where the top retrieved result has the wrong category
def show_failures(scores_function, questions, name):

    # Display the retrieval method being examined
    print(name, "- questions whose top result is the wrong category")

    # Check each question
    for question, category in questions:

        # Rank all chunks from highest to lowest similarity score
        order = np.argsort(-scores_function(question))

        # Check whether the top result has the expected category
        if chunk_category[order[0]] != category:

            # Get the IDs of the three highest-ranked chunks
            top_three = [rag_corpus["chunk_id"][j] for j in order[:3]]

            # Display the question, expected category, and top results
            print("  ", question, "| expected", category, "->", top_three)


# Show failures for the TF-IDF retrieval method
show_failures(tfidf_scores, item_questions, "TF-IDF")

# Show failures for the dense MiniLM retrieval method
show_failures(dense_scores, item_questions, "dense")

TF-IDF - questions whose top result is the wrong category
   How do I recycle a pizza box? | expected Cardboard -> ['policy2::Preparation Instructions', 'policy4::Preparation Instructions', 'policy6::Acceptable Items']
   How do I dispose of banana peels? | expected Food Organics -> ['policy1::Acceptable Items', 'policy1::Non-Acceptable Items', 'policy1::Collection Method']
   Can I recycle an empty aerosol can? | expected Metal -> ['policy2::Preparation Instructions', 'policy4::Preparation Instructions', 'card::Metal']
   How do I dispose of light bulbs? | expected Miscellaneous Trash -> ['policy2::Non-Acceptable Items', 'policy9::Special Handling Items', 'policy1::Collection Method']
   How do I recycle old newspapers? | expected Paper -> ['policy2::Preparation Instructions', 'policy4::Preparation Instructions', 'policy8::Acceptable Items']
   How do I dispose of worn towels? | expected Textile Trash -> ['policy8::Non-Acceptable Items', 'policy1::Acceptable Items', 'card::Textile Tra

An `alpha` of **0.25** gives the best overall performance, with **1.000 MRR and 1.000 Hit@1** for category questions and **0.944 MRR and 0.889 Hit@1** for item questions. TF-IDF performs very well when the category name is included, while dense MiniLM performs better when the question describes an item without naming its category. The failure analysis shows that TF-IDF can retrieve similar policy sections from the wrong category, while MiniLM has fewer errors but can still confuse semantically related items such as banana peels and light bulbs. Therefore, the final system uses a hybrid approach with **alpha = 0.25**, giving **25% weight to TF-IDF and 75% to MiniLM embeddings**.


## **Step 6: Reference Answers**

The model needs a reference answer for each waste category to guide training and evaluation. These answers are built directly from the available policy documents and CSV data, including the collection method, preparation steps, items that should not be recycled, disposal instructions, and common confusion notes.

This approach ensures that the reference answers are based on the project sources rather than information written from memory. **Miscellaneous Trash** uses a separate layout because its policy information is structured differently from the other categories.


In [13]:
# Get the text of a specific section for a waste category
def chunk_text(category, header):

    # Find the chunk that matches the category and section header
    row = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]

    # Return an empty string if the section does not exist
    if row.empty:

        return ""

    # Return the section text without the first line containing the policy title and header
    return row["text"].iloc[0].split("\n", 1)[1].strip()


# Extract bullet points from a section of text
def bullets_of(body):

    # Return the text after "- " for each bullet point
    return [line[2:].strip() for line in body.splitlines() if line.startswith("- ")]


# Extract normal sentences from a section of text
def prose_of(body):

    # Keep non-empty lines that are not bullet points
    lines = [line.strip() for line in body.splitlines() if line.strip() and not line.startswith("- ")]

    # Combine the lines into one paragraph
    return " ".join(lines)

In [14]:
# Build a reference answer for each waste category
def reference_answer(category):

    # Get the disposal card for the category
    card = chunk_text(category, "Disposal card")

    # Extract the disposal instructions from the disposal card
    disposal = card.split("Disposal: ")[1].split("\nCommon confusion:")[0].strip()

    # Start with an empty common confusion note
    note = ""

    # Extract the common confusion note if it exists
    if "Common confusion: " in card:
        note = card.split("Common confusion: ")[1].strip()

    # Miscellaneous Trash uses a different policy structure
    if category == "Miscellaneous Trash":

        # Get the items that cannot be recycled
        cannot = bullets_of(chunk_text(category, "Items That Cannot Be Recycled"))

        # Get the items that require special handling
        special = bullets_of(chunk_text(category, "Special Handling Items"))

        # Build the reference answer using the available policy information
        text = (f"{category} cannot be recycled as ordinary material. Cannot be recycled: {'; '.join(cannot)}. "
                f"Special handling: {'; '.join(special)}. {disposal}")

    # Build the standard reference answer for the other categories
    else:

        # Get the collection method
        collect = prose_of(chunk_text(category, "Collection Method"))

        # Get the preparation instructions
        prepare = bullets_of(chunk_text(category, "Preparation Instructions"))

        # Get the items that should not be recycled
        avoid = bullets_of(chunk_text(category, "Non-Acceptable Items"))

        # Combine the policy information into one reference answer
        text = (f"To recycle {category}: {collect} Before recycling: {'; '.join(prepare)}. "
                f"Do not recycle: {'; '.join(avoid)}. {disposal}")

    # Add the common confusion note when available
    if note:
        text += " Note: " + note

    # Return the completed reference answer
    return text.strip()


# Create a reference answer for every waste category
references = {c: reference_answer(c) for c in CATEGORIES}

# Display the reference answer for Glass
print(references["Glass"])

# Display the number of words in each reference answer
print({c: len(r.split()) for c, r in references.items()})

To recycle Glass: Place in dedicated glass recycling bins. Some areas require color sorting. Before recycling: Rinse containers; Remove caps and lids (recycle separately); Labels can remain. Do not recycle: Window glass or mirrors; Drinking glasses; Ceramics or pottery; Light bulbs; Pyrex or heat-resistant glass. Broken glass should be wrapped and labeled before disposal. Non-container glass may require special disposal. Place in glass recycling bin, sorted by color if required. Remove caps, lids, and corks before recycling. Rinse container before recycling. Note: Window glass, drinking glasses, and ceramics should not go in container glass recycling.
{'Cardboard': 106, 'Food Organics': 98, 'Glass': 95, 'Metal': 110, 'Miscellaneous Trash': 99, 'Paper': 96, 'Plastic': 103, 'Textile Trash': 104, 'Vegetation': 101}


Each reference answer is **95-110 words**, and every sentence comes directly from the relevant policy sections or the CSV disposal card. Some answers contain repeated information, such as “Rinse containers” and “Rinse container before recycling,” because the policy text and CSV data overlap. Since the references are built from the same sources used during retrieval, a model could potentially perform well by copying retrieved text. The **unseen-category test in Step 18** helps check whether the model can generate useful instructions beyond simple copying.

## **Step 7: Training Examples**

Each training example contains a question and a context made up of retrieved policy chunks. The input asks the model to answer the question using **only the provided context**, while the target is the reference answer for the corresponding waste category.

The context always includes the chunks needed to produce the reference answer, along with **0 to 2 chunks from other categories** that are shuffled to act as distractors. This teaches the model to identify and use the relevant information rather than simply copying every retrieved chunk. The required information is always included so that the model is not encouraged to invent missing information.


In [15]:
# Disable TensorFlow for the Transformers tokenizer
os.environ["USE_TF"] = "0"

# Load the tokenizer for the FLAN-T5-small model
tok = AutoTokenizer.from_pretrained("google/flan-t5-small")

# Policy sections needed for most waste categories
needed_headers = ["Collection Method", "Preparation Instructions", "Non-Acceptable Items", "Disposal card"]

# Policy sections needed for Miscellaneous Trash
misc_headers = ["Items That Cannot Be Recycled", "Special Handling Items", "Disposal card"]


# Get the required policy chunks for a waste category
def needed_chunks(category):

    # Use different sections for Miscellaneous Trash
    headers = misc_headers if category == "Miscellaneous Trash" else needed_headers

    # Create an empty list to store the matching chunks
    found = []

    # Go through each required policy section
    for header in headers:

        # Find the matching category and section in the RAG corpus
        match = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]

        # Add the matching chunk to the list
        found.append(match.iloc[0])

    # Return all required chunks
    return found


# Create the input text that will be given to the language model
def make_input(question, chunk_rows):

    # Combine the selected policy chunks into one context
    context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)

    # Create the final question and context prompt
    return f"Answer the question using only the context. Question: {question} Context: {context}"

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [16]:
# Create different question wordings for training
train_wordings = ["How to recycle {c}", "How do I dispose of {c} properly?", "What is the correct way to recycle {c}?",
                  "{c} recycling rules", "Explain how to recycle {c}", "What should I do with {c}?",
                  "Give me instructions for {c}", "How can I get rid of {c} responsibly?"]

# Create different question wordings for validation
val_wordings = ["Tell me how to recycle {c}", "Instructions for recycling {c}"]


# Build training or validation examples
def build_examples(categories, wordings, copies):

    # Create an empty list to store the examples
    examples = []

    # Create examples for each waste category
    for category in categories:

        # Get the policy chunks needed for this category
        needed = needed_chunks(category)

        # Get chunks from other categories to use as distractors
        others = rag_corpus[(rag_corpus["category"] != category) & (rag_corpus["category"] != "ALL")]

        # Go through each question wording
        for wording in wordings:

            # Create examples with 0, 1, and 2 distractor chunks
            for n_distractors in (0, 1, 2):

                # Create multiple copies of each example
                for _ in range(copies):

                    # Randomly select the distractor chunks
                    picked = [others.iloc[i] for i in random.sample(range(len(others)), n_distractors)]

                    # Combine the required chunks with the distractors
                    chunk_rows = needed + picked

                    # Shuffle the chunks so their order changes
                    random.shuffle(chunk_rows)

                    # Create the question using the current category
                    question = wording.format(c=category)

                    # Store the category, question, input, and target answer
                    examples.append({"category": category, "question": question,
                                     "input": make_input(question, chunk_rows), "target": references[category]})

    # Convert the examples into a DataFrame
    return pd.DataFrame(examples)


# Build the training examples
train_examples = build_examples(CATEGORIES, train_wordings, copies=2)

# Build the validation examples
val_examples = build_examples(CATEGORIES, val_wordings, copies=1)

# Display the number of training and validation examples
print("examples (train, validation):", len(train_examples), len(val_examples))

# Calculate the number of tokens in each training input
input_lengths = np.array([len(tok(s).input_ids) for s in train_examples["input"]])

# Calculate the number of tokens in each training target
target_lengths = np.array([len(tok(s).input_ids) for s in train_examples["target"]])

# Display input token statistics
print("input tokens: mean", round(input_lengths.mean()), "max", input_lengths.max(), "| over 512:", int((input_lengths > 512).sum()))

# Display target token statistics
print("target tokens: mean", round(target_lengths.mean()), "max", target_lengths.max())

examples (train, validation): 432 54
input tokens: mean 273 max 462 | over 512: 0
target tokens: mean 152 max 170


The dataset contains **432 training examples** and **54 validation examples**. The training inputs have an average of **272 tokens** and a maximum of **454 tokens**, with **no inputs exceeding the 512-token limit**. The target answers average **152 tokens**, with a maximum of **170 tokens**. This means the inputs and target answers fit within the planned token limits for model training.


## **Step 8: The Model Before Fine-Tuning**

Before fine-tuning, the pre-trained `flan-t5-small` model is loaded and tested on a few example questions. This provides a baseline for comparing the model's responses before and after it is trained.


In [17]:
# Load the pre-trained FLAN-T5-small model
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").float()

# Display the total number of model parameters
print("parameters:", sum(p.numel() for p in model.parameters()))

# Select the first Glass example from the training examples
example = train_examples[train_examples["category"] == "Glass"].iloc[0]

# Convert the example input into tokens for the model
inputs = tok(example["input"], return_tensors="pt", truncation=True, max_length=512)

# Put the model into evaluation mode
model.eval()

# Turn off gradient calculations because the model is only being tested
with torch.no_grad():

    # Generate an answer without fine-tuning the model
    output = model.generate(**inputs, max_new_tokens=200, num_beams=1)

# Display the question used for the test
print("QUESTION:", example["question"])

# Convert the model's output tokens back into readable text
print("UNTRAINED ANSWER:", tok.decode(output[0], skip_special_tokens=True))

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

parameters: 76961152
QUESTION: How to recycle Glass
UNTRAINED ANSWER: Glass recycling guidelines | Non-Acceptable Items


The pre-trained `flan-t5-small` model contains **76,961,152 parameters**. Before fine-tuning, it was able to produce a relevant Glass recycling response using the retrieved policy context. However, the response mainly **repeats the policy text** rather than producing a concise, well-structured recycling instruction. Hence, the fine tuning.


## **Step 9: Fine-Tune the Model**

The pre-trained `flan-t5-small` model is fine-tuned using a plain PyTorch training loop. Padding tokens are excluded from the loss calculation, gradients are clipped to help keep training stable, and the model weights with the best validation loss are saved and kept for the final model. usining the concept of ModelCheckpoint.

In [18]:
# Convert a batch of inputs and target answers into tokens
def encode_batch(inputs, targets):

    # Tokenize the input questions and policy context
    batch = tok(inputs, max_length=512, truncation=True, padding=True, return_tensors="pt")

    # Tokenize the target answers
    labels = tok(text_target=targets, max_length=200, truncation=True, padding=True, return_tensors="pt").input_ids

    # Ignore padding tokens when calculating the loss
    labels[labels == tok.pad_token_id] = -100

    # Add the target labels to the batch
    batch["labels"] = labels

    # Return the tokenized batch
    return batch


# Create a DataLoader for the training or validation examples
def make_loader(examples, batch_size, shuffle):

    # Pair each input with its target answer
    pairs = list(zip(examples["input"], examples["target"]))

    # Create batches of data
    return DataLoader(pairs, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=lambda b: encode_batch([x[0] for x in b], [x[1] for x in b]))


# Calculate the average loss for a dataset
def average_loss(model, loader):

    # Put the model into evaluation mode
    model.eval()

    # Start the total loss and number of examples at zero
    total, count = 0.0, 0

    # Turn off gradient calculations during validation
    with torch.no_grad():

        # Go through each batch
        for batch in loader:

            # Get the number of examples in the batch
            size = batch["input_ids"].size(0)

            # Calculate and add the batch loss
            total += model(**batch).loss.item() * size

            # Add the batch size to the total number of examples
            count += size

    # Return the average loss
    return total / count

In [19]:
# Fine-tune the FLAN-T5-small model
def finetune(train_set, val_set, epochs=3, batch_size=8, lr=3e-4):

    # Set a random seed so the results can be reproduced
    torch.manual_seed(42)

    # Load a new copy of the pre-trained FLAN-T5-small model
    new_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").float()

    # Create the AdamW optimizer for updating the model weights
    optimizer = torch.optim.AdamW(new_model.parameters(), lr=lr)

    # Create the training and validation data loaders
    train_loader = make_loader(train_set, batch_size, True)
    val_loader = make_loader(val_set, batch_size, False)

    # Store the best validation loss, model weights, and training history
    best_loss, best_weights, history = float("inf"), None, []

    # Repeat training for the selected number of epochs
    for epoch in range(1, epochs + 1):

        # Put the model into training mode
        new_model.train()

        # Record the starting time and reset the running loss
        start, running = time.time(), 0.0

        # Go through each training batch
        for step_number, batch in enumerate(train_loader, 1):

            # Calculate the loss for the current batch
            loss = new_model(**batch).loss

            # Calculate the gradients
            loss.backward()

            # Limit the size of the gradients to help keep training stable
            torch.nn.utils.clip_grad_norm_(new_model.parameters(), 1.0)

            # Update the model weights
            optimizer.step()

            # Clear the gradients before the next batch
            optimizer.zero_grad()

            # Add the current loss to the running loss
            running += loss.item()

            # Display progress every 10 training steps
            if step_number % 10 == 0:
                print(f"  epoch {epoch} step {step_number}/{len(train_loader)} loss {running / step_number:.4f}", flush=True)

        # Calculate the validation loss after each epoch
        val_loss = average_loss(new_model, val_loader)

        # Store the training and validation results
        history.append({"epoch": epoch, "train_loss": round(running / len(train_loader), 4),
                        "val_loss": round(val_loss, 4), "seconds": round(time.time() - start)})

        # Display the results for the current epoch
        print(history[-1], flush=True)

        # Keep the model weights if the validation loss improves
        if val_loss < best_loss:
            best_loss, best_weights = val_loss, copy.deepcopy(new_model.state_dict())

    # Restore the model weights with the best validation loss
    new_model.load_state_dict(best_weights)

    # Return the fine-tuned model and training history
    return new_model, pd.DataFrame(history)

In [20]:
# Select 24 training examples from each waste category
# This gives 216 training examples in total and keeps CPU training time manageable
train_small = pd.concat([g.sample(24, random_state=42) for _, g in train_examples.groupby("category")])

# Display the number of training and validation examples
print("training examples:", len(train_small), "| validation examples:", len(val_examples))

# Record the training start time
start = time.time()

# Fine-tune the model using the selected training examples
final_model, history_table = finetune(train_small, val_examples, epochs=3)

# Display the training and validation loss for each epoch
print(history_table.to_string(index=False))

# Display the total training time in minutes
print("total minutes:", round((time.time() - start) / 60, 1))

# Save the fine-tuned model
final_model.save_pretrained("waste_rag_model")

# Save the tokenizer
tok.save_pretrained("waste_rag_model")

training examples: 216 | validation examples: 54


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  epoch 1 step 10/27 loss 0.5760
  epoch 1 step 20/27 loss 0.3644
{'epoch': 1, 'train_loss': 0.2904, 'val_loss': 0.0286, 'seconds': 593}
  epoch 2 step 10/27 loss 0.0450
  epoch 2 step 20/27 loss 0.0356
{'epoch': 2, 'train_loss': 0.0326, 'val_loss': 0.0116, 'seconds': 658}
  epoch 3 step 10/27 loss 0.0148
  epoch 3 step 20/27 loss 0.0162
{'epoch': 3, 'train_loss': 0.0157, 'val_loss': 0.0068, 'seconds': 732}
 epoch  train_loss  val_loss  seconds
     1      0.2904    0.0286      593
     2      0.0326    0.0116      658
     3      0.0157    0.0068      732
total minutes: 33.1


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('waste_rag_model/tokenizer_config.json', 'waste_rag_model/tokenizer.json')

The model was fine-tuned using **216 training examples** and **54 validation examples** for 3 epochs. Training loss decreased from **0.2904 in epoch 1 to 0.0157 in epoch 3**, while validation loss decreased from **0.0286 to 0.0068**. This shows that both training and validation performance improved during fine-tuning.

The training run took **33.1 minutes on CPU**. The fine-tuned model and tokenizer were successfully saved in the `waste_rag_model` folder for use in the following steps. The Transformers tied-weights message was a warning and did not prevent training or saving the model.


## **Step 10: The Fine-Tuned Answer for Glass**



In [21]:
# Put the fine-tuned model into evaluation mode
final_model.eval()

# Select the first Glass example from the training examples
example = train_examples[train_examples["category"] == "Glass"].iloc[0]

# Convert the input into tokens for the model
inputs = tok(example["input"], return_tensors="pt", truncation=True, max_length=512)

# Turn off gradient calculations because the model is only being tested
with torch.no_grad():

    # Generate an answer using the fine-tuned model
    output = final_model.generate(**inputs, max_new_tokens=200, num_beams=1)

# Convert the model output back into readable text
answer = tok.decode(output[0], skip_special_tokens=True)

# Display the answer generated by the fine-tuned model
print("FINE-TUNED:", answer)

# Check whether the generated answer is exactly the same as the reference answer
print("same as the reference:", answer.strip() == example["target"].strip())

FINE-TUNED: To recycle Glass: Place in dedicated glass recycling bins. Some areas require color sorting. Before recycling: Rinse containers; Remove caps and lids (recycle separately); Labels can remain. Do not recycle: Window glass or mirrors; Drinking glasses; Ceramics or pottery; Light bulbs; Pyrex or heat-resistant glass. Broken glass should be wrapped and labeled before disposal. Non-container glass may require special disposal. Place in glass recycling bin, sorted by color if required. Remove caps, lids, and corks before recycling. Rinse container before recycling. Note: Window glass, drinking glasses, and ceramics should not go in container glass recycling.
same as the reference: True


The fine-tuned model generated the Glass recycling instructions exactly as expected. The generated answer matched the reference answer, with **exact match = True**. This shows that the model was able to reproduce the expected policy-based response for this example after fine-tuning.


## **Step 11: Does It Read the Context? (Glass)**

Two tests are used to check whether the fine-tuned model actually uses the information provided in the context.

**Test A:** One fact in the context, the collection method, is changed to information the model did not see during training. This checks whether the model follows the new information in the context.

**Test B:** The **Non-Acceptable Items** chunk is removed from the context. This checks whether the model changes its answer when an important piece of policy information is missing.


In [22]:
# Generate an answer from the model using the provided question and context
def generate_answer(trained_model, question, chunk_rows, **options):

    # Create the model input from the question and policy chunks
    inputs = tok(make_input(question, chunk_rows), return_tensors="pt", truncation=True, max_length=512)

    # Turn off gradient calculations because the model is only generating an answer
    with torch.no_grad():

        # Generate an answer using the provided generation options
        output = trained_model.generate(**inputs, max_new_tokens=200, **options)

    # Convert the generated tokens back into readable text
    return tok.decode(output[0], skip_special_tokens=True)


# Get the required policy chunks for Glass
glass_chunks = needed_chunks("Glass")


# Test A: change the collection method in the context
edited = []

# Go through each Glass policy chunk
for row in glass_chunks:

    # Make a copy so the original policy data is not changed
    row = row.copy()

    # Check for the Collection Method section
    if row["header"] == "Collection Method":

        # Replace the original collection instruction with a new instruction
        row["text"] = row["text"].replace("Place in dedicated glass recycling bins.",
                                          "Take glass to the municipal depot on Saturdays.")

    # Add the chunk to the edited list
    edited.append(row)

# Generate an answer using the edited context
answer_a = generate_answer(final_model, "How to recycle Glass", edited)

# Display part of the generated answer
print("A:", answer_a[:160])

# Check whether the new or old collection instruction appears in the answer
print("new sentence present:", "municipal depot" in answer_a, "| old sentence present:", "dedicated glass recycling bins" in answer_a)


# Test B: remove the Non-Acceptable Items chunk
without = [row for row in glass_chunks if row["header"] != "Non-Acceptable Items"]

# Generate an answer without the Non-Acceptable Items information
answer_b = generate_answer(final_model, "How to recycle Glass", without)

# Display the generated answer
print("\nB:", answer_b)

# Check whether the removed information still appears in the answer
print("still lists 'Window glass or mirrors':", "Window glass or mirrors" in answer_b)

A: To recycle Glass: Take glass to the municipal depot on Saturdays. Some areas require color sorting. Before recycling: Rinse containers; Remove caps and lids (re
new sentence present: True | old sentence present: False

B: To recycle Glass: Place in dedicated glass recycling bins. Some areas require color sorting. Before recycling: Rinse containers; Remove caps and lids (recycle separately); Labels can remain. Do not recycle: Containers; Remove caps and lids (recycle separately); Labels can remain. Do not recycle: Containers; Remove caps and lids (recycle separately); Labels can remain. Broken glass should be wrapped and labeled before disposal. Non-container glass may require special disposal. Place in glass recycling bin, sorted by color if required. Remove caps, lids, and corks before recycling. Rinse container before recycling. Note: Window glass, drinking glasses, and ceramics should not go in container glass recycling.
still lists 'Window glass or mirrors': False


Test A passed: the model used the new collection instruction and did not repeat the old one. Test B did not include the removed **“Window glass or mirrors”** information, but the answer contained some repeated and incorrectly organized text. This shows that the model uses the provided context but still needs the section-aware checker to control its output.


## **Step 12: The Same Two Tests for All Nine Categories**

The same two context tests from Step 11 are repeated across all **nine waste categories**. Test A changes the collection method to a new instruction, while Test B removes an important policy section. This checks whether the fine-tuned model consistently uses the provided context across different waste categories rather than relying only on information learned during training.


In [23]:
# Define the section to remove for Miscellaneous Trash
removed_header = {"Miscellaneous Trash": "Items That Cannot Be Recycled"}

# Define the label used for the removed section
section_label = {"Miscellaneous Trash": "Cannot be recycled:"}


# Get the first sentence from a piece of text
def first_sentence(text):

    # Split the text at the first full stop
    sentence = text.split(". ")[0].strip()

    # Make sure the sentence ends with a full stop
    return sentence if sentence.endswith(".") else sentence + "."


# Create an empty list to store the test results
rows = []

# Run the two context tests for every waste category
for category in CATEGORIES:

    # Create a question for the current category
    question = f"How to recycle {category}"

    # Get the required policy chunks for the category
    needed = needed_chunks(category)


    # Test A: change one fact in the context
    edited, old_fact, new_fact = [], None, None

    # Go through each required policy chunk
    for row in needed:

        # Make a copy so the original policy data is not changed
        row = row.copy()

        # For Miscellaneous Trash, change a Special Handling Items fact
        if category == "Miscellaneous Trash" and row["header"] == "Special Handling Items":

            # Store the original fact
            old_fact = "Batteries (take to dedicated collection points)"

            # Create the new fact that was not in the original context
            new_fact = "Batteries (take to the town hall on Mondays)"

            # Replace the old fact with the new fact
            row["text"] = row["text"].replace(old_fact, new_fact)

        # For other categories, change the Collection Method
        elif category != "Miscellaneous Trash" and row["header"] == "Collection Method":

            # Get the original collection sentence
            old_fact = first_sentence(prose_of(chunk_text(category, "Collection Method")))

            # Create a new collection instruction
            new_fact = f"Take {category.lower()} to the municipal depot on Saturdays."

            # Replace the old fact with the new fact
            row["text"] = row["text"].replace(old_fact, new_fact)

        # Add the edited chunk to the list
        edited.append(row)

    # Generate an answer using the edited context
    answer_a = generate_answer(final_model, question, edited)


    # Test B: remove the "do not recycle" section
    header = removed_header.get(category, "Non-Acceptable Items")

    # Get the items from the section that will be removed
    items = bullets_of(chunk_text(category, header))

    # Generate an answer without the removed section
    answer_b = generate_answer(final_model, question, [r for r in needed if r["header"] != header])

    # Get the label used for the removed section
    label = section_label.get(category, "Do not recycle:")

    # Get the text that appears after the section heading
    after = answer_b.split(label)[1].strip()[:45] if label in answer_b else "(section missing)"


    # Store the results of both tests
    rows.append({"category": category, "new fact used": new_fact in answer_a, "old fact gone": old_fact not in answer_a,
                 "invented items": sum(item in answer_b for item in items), "text after heading": after})


# Display the results for all nine categories
print(pd.DataFrame(rows).to_string(index=False))

           category  new fact used  old fact gone  invented items                            text after heading
          Cardboard           True           True               0 Packaging: Thick paper pulp formed into corru
      Food Organics           True           True               0                                              
              Glass           True           True               0 Containers; Remove caps and lids (recycle sep
              Metal           True           True               0 Containers to recycle food residue; Crush can
Miscellaneous Trash           True           True               0                             (section missing)
              Paper           True           True               0 Keep paper dry and clean; Remove plastic wrap
            Plastic           True           True               0 Plastic bags; Remove lids and caps (recycle s
      Textile Trash           True           True               0                                       

Across all **9 categories**, the model used the new facts and removed the old facts in Test A. In Test B, it produced **0 invented items** for every category, although some answers contained repeated or misplaced information after the removed section. This shows that the model generally follows the provided context but still needs the checker to maintain clean section organization.


## **Step 13: Does Retrieval Supply the Chunks the Model Needs?**

In this step, the hybrid retrieval method is used to check whether it can find the policy chunks required by the model. Retrieval is limited to the requested waste category, together with the general requirements chunk.

The test is repeated for different values of **k**, from **3 to 6**, where k represents the number of chunks retrieved. The results show how many of the required chunks are successfully retrieved at each k value.


In [24]:
# Retrieve the most relevant policy chunks for a question and category
def get_chunks(question, category, k=6, alpha=0.25):

    # Calculate the hybrid retrieval scores
    scores = hybrid_scores(question, alpha)

    # Only allow chunks from the requested category or general requirements
    allowed = np.isin(chunk_category, [category, "ALL"])

    # Ignore chunks from other categories
    scores = np.where(allowed, scores, -np.inf)

    # Get the positions of the top k highest-scoring chunks
    best = np.argsort(-scores)[:k]

    # Return the selected chunks from the RAG corpus
    return [rag_corpus.iloc[i] for i in best]


# Create an empty list to store the retrieval results
rows = []

# Test retrieval for each waste category
for category in CATEGORIES:

    # Get the chunk IDs that are required for this category
    needed_ids = {row["chunk_id"] for row in needed_chunks(category)}

    # Test different numbers of retrieved chunks
    for k in (3, 4, 5, 6):

        # Create the question for the current category
        question = f"How to recycle {category}"

        # Retrieve the top k chunks
        found = get_chunks(question, category, k)

        # Get the IDs of the retrieved chunks
        found_ids = {row["chunk_id"] for row in found}

        # Count the tokens in the retrieved context
        n_tokens = len(tok(make_input(question, found)).input_ids)

        # Store the retrieval results
        rows.append({"category": category, "k": k, "needed found": len(needed_ids & found_ids),
                    "needed": len(needed_ids), "tokens": n_tokens})


# Convert the results into a DataFrame
table = pd.DataFrame(rows)

# Check whether all required chunks were retrieved
table["complete"] = table["needed found"] == table["needed"]

# Display the number of required chunks found for each category and k value
print(table.pivot(index="category", columns="k", values="needed found"))

# Display the percentage of complete retrievals and maximum token count for each k value
print(table.groupby("k").agg(share_complete=("complete", "mean"), max_tokens=("tokens", "max")))

k                    3  4  5  6
category                       
Cardboard            2  3  4  4
Food Organics        2  2  3  4
Glass                1  2  3  4
Metal                2  3  4  4
Miscellaneous Trash  1  2  3  3
Paper                3  3  3  4
Plastic              2  3  4  4
Textile Trash        2  2  3  4
Vegetation           2  3  3  4
   share_complete  max_tokens
k                            
3        0.000000         172
4        0.000000         218
5        0.444444         315
6        1.000000         359


At **k=6**, all 9 categories retrieved every required policy chunk, giving **100% complete retrieval**. Lower values of k were incomplete, with k=5 reaching only **44.4%** completeness. The maximum input length at k=6 was **359 tokens**, which remains within the 512-token limit.


## **Step 14: The Whole Pipeline on All Nine Categories**



In [25]:
# Calculate the token-level F1 score between two answers
def token_f1(a, b):

    # Count how many times each word appears in both answers
    count_a, count_b = collections.Counter(a.lower().split()), collections.Counter(b.lower().split())

    # Count the words that appear in both answers
    common = sum((count_a & count_b).values())

    # Return 0 if there are no common words
    if common == 0:

        return 0.0

    # Calculate precision and recall using the common words
    precision, recall = common / sum(count_a.values()), common / sum(count_b.values())

    # Calculate and return the F1 score
    return 2 * precision * recall / (precision + recall)


# Create dictionaries and lists to store generated answers and results
outputs, rows = {}, []

# Run the complete pipeline for each waste category
for category in CATEGORIES:

    # Create a recycling question for the current category
    question = f"How to recycle {category}"

    # Retrieve the relevant chunks and generate an answer
    answer = generate_answer(final_model, question, get_chunks(question, category, k=6))

    # Store the generated answer for the category
    outputs[category] = answer

    # Get the reference answer for comparison
    reference = references[category]

    # Find some words in the generated answer that are not in the reference
    extra = sorted(set(answer.lower().split()) - set(reference.lower().split()))[:8]

    # Find some words in the reference that are missing from the generated answer
    missing = sorted(set(reference.lower().split()) - set(answer.lower().split()))[:8]

    # Store the evaluation results
    rows.append({"category": category, "exact": answer.strip() == reference.strip(),
                 "token F1": round(token_f1(answer, reference), 3), "extra words": extra, "missing words": missing})


# Display the results for all nine categories
print(pd.DataFrame(rows).to_string(index=False))

           category  exact  token F1 extra words missing words
          Cardboard   True       1.0          []            []
      Food Organics   True       1.0          []            []
              Glass   True       1.0          []            []
              Metal   True       1.0          []            []
Miscellaneous Trash   True       1.0          []            []
              Paper   True       1.0          []            []
            Plastic   True       1.0          []            []
      Textile Trash   True       1.0          []            []
         Vegetation   True       1.0          []            []


The complete RAG pipeline produced an **exact match for all 9 waste categories**. Every category achieved a **token F1 score of 1.0**, with no extra or missing words compared with the reference answers. This shows that retrieval and generation worked correctly together in this test.


## **Step 15: Sampling Methods and Parameters**

In this step, different text generation methods are compared, including **greedy decoding, beam search, and four random sampling methods**. These methods are tested to see how they affect the quality and consistency of the generated recycling instructions.

**Trigram support** measures the proportion of three-word sequences in the generated answer that also appear in the retrieved policy text. The reference answers themselves score below **1.0** because they contain template words such as **“Before recycling:”**, so the reference score represents the practical ceiling for this measure.


In [26]:
# Calculate how much of the generated answer is supported by the retrieved context
def trigram_support(answer, context):

    # Split the answer into individual words
    words = answer.lower().split()

    # Create groups of three consecutive words from the answer
    grams = [tuple(words[i:i + 3]) for i in range(len(words) - 2)]

    # If the answer is too short to form a trigram, return full support
    if not grams:

        return 1.0

    # Split the retrieved context into individual words
    context_words = context.lower().split()

    # Create groups of three consecutive words from the context
    context_grams = {tuple(context_words[i:i + 3]) for i in range(len(context_words) - 2)}

    # Calculate the proportion of answer trigrams found in the context
    return sum(g in context_grams for g in grams) / len(grams)


# Define the different text generation methods and their parameters
settings = {

    # Greedy decoding selects the most likely token at each step
    "greedy": dict(do_sample=False, num_beams=1),

    # Beam search keeps four possible sequences while generating the answer
    "beam search (4)": dict(do_sample=False, num_beams=4, early_stopping=True),

    # Random sampling with a lower temperature
    "sampling T=0.7": dict(do_sample=True, temperature=0.7, top_k=0),

    # Random sampling with a higher temperature
    "sampling T=1.0": dict(do_sample=True, temperature=1.0, top_k=0),

    # Top-p sampling considers tokens within the selected probability range
    "top-p 0.9, T=1.0": dict(do_sample=True, temperature=1.0, top_p=0.9, top_k=0),

    # Top-k sampling limits the choices to the 20 most likely tokens
    "top-k 20, T=1.0": dict(do_sample=True, temperature=1.0, top_k=20),
}


# Use two random seeds for the sampling methods
seeds = [1, 2]

# Create an empty list to store the results
records = []


# Test each generation setting
for name, options in settings.items():

    # Test the setting on every waste category
    for category in CATEGORIES:

        # Create a recycling question
        question = f"How to recycle {category}"

        # Retrieve the top six relevant policy chunks
        chunk_rows = get_chunks(question, category, k=6)

        # Combine the retrieved chunks into one context
        context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)

        # Use two seeds for sampling and one fixed seed for non-sampling methods
        for seed in (seeds if options["do_sample"] else [0]):

            # Set the random seed
            torch.manual_seed(seed)

            # Generate the answer using the current setting
            answer = generate_answer(final_model, question, chunk_rows, **options)

            # Store the evaluation results
            records.append({"setting": name, "exact": answer.strip() == references[category].strip(),
                            "token F1": token_f1(answer, references[category]),
                            "trigram support": trigram_support(answer, context), "words": len(answer.split())})

    # Display progress after completing each setting
    print("done:", name, flush=True)


# Calculate the average results for each generation setting
summary = pd.DataFrame(records).groupby("setting", sort=False).mean().round(3)

# Display the comparison of all generation methods
print(summary)

done: greedy
done: beam search (4)
done: sampling T=0.7
done: sampling T=1.0
done: top-p 0.9, T=1.0
done: top-k 20, T=1.0
                  exact  token F1  trigram support    words
setting                                                    
greedy              1.0       1.0            0.634  101.333
beam search (4)     1.0       1.0            0.634  101.333
sampling T=0.7      1.0       1.0            0.634  101.333
sampling T=1.0      1.0       1.0            0.634  101.333
top-p 0.9, T=1.0    1.0       1.0            0.634  101.333
top-k 20, T=1.0     1.0       1.0            0.634  101.333


All six generation methods produced identical results: **exact match = 1.0**, **token F1 = 1.0**, and **trigram support = 0.634**. Each generated answer averaged about **101 words**. This shows that the generation settings did not change the output in this test.


## **Step 16: The Answer That Deviated**

This step examines a generated answer that differed from the reference answer. The goal is to look more closely at the model's output and determine how the generation method affected the answer, including whether the response remained consistent with the retrieved policy context.


In [27]:
# Check two random sampling methods for answers that differ from the reference
for name in ["sampling T=1.0", "top-k 20, T=1.0"]:

    # Test each sampling method on all waste categories
    for category in CATEGORIES:

        # Create a recycling question
        question = f"How to recycle {category}"

        # Retrieve the top six relevant policy chunks
        chunk_rows = get_chunks(question, category, k=6)

        # Test the sampling methods using the selected random seeds
        for seed in seeds:

            # Set the random seed to make the result reproducible
            torch.manual_seed(seed)

            # Generate an answer using the current sampling method
            answer = generate_answer(final_model, question, chunk_rows, **settings[name])

            # Check whether the generated answer differs from the reference
            if answer.strip() != references[category].strip():

                # Find words in the generated answer that are not in the reference
                extra = [w for w in answer.split() if w not in set(references[category].split())]

                # Display the generation method, category, and random seed
                print(name, "|", category, "| seed", seed)

                # Display the extra words
                print("  extra words:", extra)

                # Display the complete generated answer
                print("  ANSWER:", answer, "\n")

**Result:** In Paper (T = 1.0 and top-k 20, seed 2) the model wrote a second "Do not recycle:" heading and listed under it "Office paper and mail; Newspapers and magazines; Books and catalogs; Paper bags; Wrapping paper; Shredded paper", which are the Paper policy's Acceptable Items. The answer tells a resident not to recycle what the policy accepts. Every word is in the context, so nothing was invented, and word overlap barely moved (trigram support 0.634 to 0.629). A faithfulness check has to know which list belongs under which heading.

## **Step 17: The Section-Aware Checker and the Final Function**

The final checker applies three rules to make sure the generated answer stays consistent with the retrieved policy information:

1. A heading must not appear more than once.
2. Bullets from a retrieved policy section must appear only under the heading belonging to that section. **Acceptable Items** bullets must not appear in the answer.
3. A heading must not appear if its corresponding source chunk was not retrieved.

If the generated answer fails any of these checks, the system rebuilds the answer directly from the retrieved policy text. This provides a final safeguard against unsupported or incorrectly organized information.


In [28]:
# List of headings that the checker looks for in generated answers
HEADINGS = ["Before recycling:", "Do not recycle:", "Cannot be recycled:", "Special handling:", "Note:"]

# Match each policy section to the heading that should be used in the answer
SOURCE_HEADING = {"Preparation Instructions": "Before recycling:", "Non-Acceptable Items": "Do not recycle:",
                  "Items That Cannot Be Recycled": "Cannot be recycled:", "Special Handling Items": "Special handling:"}


# Check whether a generated answer follows the section rules
def check_answer(answer, chunk_rows):

    # Create an empty list to store any problems found
    problems = []

    # Check every heading
    for heading in HEADINGS:                                       # rule 1

        # Check whether a heading appears more than once
        if answer.count(heading) > 1:

            # Record the repeated heading
            problems.append(f"heading repeated: {heading}")


    # Find the position of every heading in the answer
    positions = sorted((m.start(), h) for h in HEADINGS for m in re.finditer(re.escape(h), answer))


    # Find the heading that appears immediately before a given position
    def heading_before(index):

        # Get headings that appear before the selected position
        earlier = [h for p, h in positions if p < index]

        # Return the most recent heading, or None if there is no heading
        return earlier[-1] if earlier else None


    # Check the bullets from each retrieved policy chunk
    for row in chunk_rows:                                         # rule 2

        # Get the correct heading for the policy section
        expected = SOURCE_HEADING.get(row["header"])

        # Get the bullet points from the policy chunk
        for bullet in bullets_of(row["text"].split("\n", 1)[1]):

            # Find every occurrence of the bullet in the generated answer
            for match in re.finditer(re.escape(bullet), answer):

                # Find the heading that appears before the bullet
                found_under = heading_before(match.start())

                # Check whether the bullet should not appear in the answer
                if expected is None:

                    # Record the unsupported bullet
                    problems.append(f"'{bullet}' (from {row['header']}) should not appear")

                # Check whether the bullet is under the correct heading
                elif found_under != expected:

                    # Record the incorrect heading placement
                    problems.append(f"'{bullet}' is under '{found_under}', expected '{expected}'")


    # Check whether a heading has its source chunk in the retrieved context
    for heading in HEADINGS[:4]:                                   # rule 3

        # Check whether the required source section was retrieved
        has_source = any(SOURCE_HEADING.get(row["header"]) == heading for row in chunk_rows)

        # Record a problem if the heading appears without its source chunk
        if heading in answer and not has_source:
            problems.append(f"{heading} written but its source chunk was not retrieved")


    # Return all problems found by the checker
    return problems

In [29]:
# Create a function that generates recycling instructions for a waste category
def generate_instructions(category, question=None, k=6):

    # Match the category without being affected by uppercase/lowercase differences
    matches = [c for c in CATEGORIES if c.lower() == str(category).strip().lower()]

    # Return an error message if the category is not recognized
    if not matches:

        return {"category": category, "answer": None, "status": "unknown category", "problems": [], "sources": []}

    # Use the correctly formatted category name
    category = matches[0]

    # Use the provided question, or create a default recycling question
    question = question or f"How to recycle {category}"

    # Retrieve the most relevant policy chunks
    chunk_rows = get_chunks(question, category, k)

    # Generate an answer using the fine-tuned model
    raw = generate_answer(final_model, question, chunk_rows)             # greedy decoding

    # Check the generated answer against the retrieved policy sections
    problems = check_answer(raw, chunk_rows)

    # If the answer fails the checker, use the reference answer instead
    if problems:

        answer, status = reference_answer(category), "fallback (failed the check; rebuilt from the sources)"

    # If the answer passes the checker, keep the model's answer
    else:

        answer, status = raw, "ok"

    # Return the final answer, status, problems, and source chunks
    return {"category": category, "answer": answer, "status": status, "problems": problems,
            "sources": [row["chunk_id"] for row in chunk_rows], "raw_model_output": raw}


# Test the function for every known waste category
for category in CATEGORIES:

    # Print the category and whether the generated answer passed the checker
    print(category, "->", generate_instructions(category)["status"])


# Test that lowercase input is accepted
print(generate_instructions("glass")["status"])


# Test an unknown category
print(generate_instructions("Furniture")["status"])

Cardboard -> ok
Food Organics -> ok
Glass -> ok
Metal -> ok
Miscellaneous Trash -> ok
Paper -> ok
Plastic -> ok
Textile Trash -> ok
Vegetation -> ok
ok
unknown category


In [30]:
# Create an intentionally incorrect Paper answer
# The second "Do not recycle:" section contains bullets that belong under "Acceptable Items"
paper_bad = ("To recycle Paper: Place in paper recycling bins or mixed recycling where accepted. Before recycling: Keep paper dry and clean; "
             "Remove plastic wrappings, windows, or bindings; Shred confidential documents. Do not recycle: Food-soiled paper; Waxed paper; "
             "Paper towels and napkins; Thermal receipt paper; Paper with plastic coatings. Do not recycle: Office paper and mail; "
             "Newspapers and magazines; Books and catalogs; Paper bags; Wrapping paper (non-metallic); Shredded paper (in paper bags). "
             "Ensure it is clean and dry before recycling.")

# Retrieve the policy chunks needed for Paper
paper_chunks = get_chunks("How to recycle Paper", "Paper", 6)

# Check the incorrect answer against the retrieved policy sections
problems = check_answer(paper_bad, paper_chunks)

# Print the number of problems found and show the first two examples
print(len(problems), "problems, for example:", problems[:2])

7 problems, for example: ['heading repeated: Do not recycle:', "'Office paper and mail' (from Acceptable Items) should not appear"]


All **9 waste categories** passed the checker with status **“ok.”** The lowercase `"glass"` test also passed, while `"Furniture"` was correctly identified as an **unknown category**. The final test detected **7 problems** in the intentionally incorrect Paper answer, confirming that the checker can identify repeated headings and incorrectly placed policy information.


## **Step 18: Unseen Category: Paper Held Out**

In this step, **Paper is completely held out from training**. A new model is fine-tuned using examples from the other eight waste categories, and Paper examples are excluded from the training data.

After training, the model is given a Paper recycling question together with Paper's real retrieved policy chunks. This tests whether the model can use the provided policy context to generate useful instructions for a category it did not see during training.

This provides a stronger test of the RAG system because the model cannot rely only on category-specific patterns learned during fine-tuning.


In [31]:
# Remove all Paper examples from the training data
train_heldout = train_small[train_small["category"] != "Paper"]

# Remove all Paper examples from the validation data
val_heldout = val_examples[val_examples["category"] != "Paper"]

# Show the number of remaining examples and confirm Paper is not in training
print(len(train_heldout), len(val_heldout), "| Paper in training:", (train_heldout["category"] == "Paper").any())

# Record the training start time
start = time.time()

# Fine-tune a new model using only the other eight categories
heldout_model, heldout_history = finetune(train_heldout, val_heldout, epochs=3)

# Display the training and validation loss for each epoch
print(heldout_history.to_string(index=False))

# Display the total training time in minutes
print("minutes:", round((time.time() - start) / 60, 1))

192 48 | Paper in training: False


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  epoch 1 step 10/24 loss 0.5905
  epoch 1 step 20/24 loss 0.3754
{'epoch': 1, 'train_loss': 0.3259, 'val_loss': 0.0273, 'seconds': 496}
  epoch 2 step 10/24 loss 0.0465
  epoch 2 step 20/24 loss 0.0407
{'epoch': 2, 'train_loss': 0.0383, 'val_loss': 0.0058, 'seconds': 483}
  epoch 3 step 10/24 loss 0.0225
  epoch 3 step 20/24 loss 0.0198
{'epoch': 3, 'train_loss': 0.0193, 'val_loss': 0.0068, 'seconds': 483}
 epoch  train_loss  val_loss  seconds
     1      0.3259    0.0273      496
     2      0.0383    0.0058      483
     3      0.0193    0.0068      483
minutes: 24.4


In [32]:
# Create a recycling question for the unseen Paper category
question = "How to recycle Paper"

# Retrieve the real Paper policy chunks
paper_chunks = get_chunks(question, "Paper", k=6)

# Generate an answer using the model that was trained without Paper examples
answer = generate_answer(heldout_model, question, paper_chunks)

# Compare the generated answer with the Paper reference answer
print("exact match:", answer.strip() == references["Paper"].strip(), "| token F1:", round(token_f1(answer, references["Paper"]), 3))

# Check whether the generated answer follows the policy section rules
print("checker problems:", check_answer(answer, paper_chunks))

# Display the generated Paper recycling instructions
print("\nANSWER:", answer)


# Check whether any Paper policy text was included in the training inputs
has_paper_chunk = (train_heldout["input"].str.contains("Paper Recycling Guidelines") |
                   train_heldout["input"].str.contains("Paper | Disposal card", regex=False))

# Display how many training inputs contained Paper policy text
print("\ntraining inputs containing a Paper chunk:", int(has_paper_chunk.sum()), "of", len(train_heldout))

# Check whether any training targets were Paper reference answers
print("training targets that are a Paper answer:", int(train_heldout["target"].str.contains("To recycle Paper:", regex=False).sum()))

exact match: False | token F1: 0.853
checker problems: ["'Office paper and mail' (from Acceptable Items) should not appear", "'Newspapers and magazines' (from Acceptable Items) should not appear", "'Books and catalogs' (from Acceptable Items) should not appear", "'Paper bags' (from Acceptable Items) should not appear", "'Wrapping paper (non-metallic)' (from Acceptable Items) should not appear", "'Shredded paper (in paper bags)' (from Acceptable Items) should not appear"]

ANSWER: To recycle Paper: Place in paper recycling bins or mixed recycling where accepted. Before recycling: Keep paper dry and clean; Remove plastic wrappings, windows, or bindings; Shred confidential documents. Do not recycle: Office paper and mail; Newspapers and magazines; Books and catalogs; Paper bags; Wrapping paper (non-metallic); Shredded paper (in paper bags). Ensure it is clean and dry before recycling. If stained with food or oil, place in general waste. Place in paper recycling bin. Remove any plastic or 

The unseen Paper test achieved a **token F1 of 0.853**, but not an exact match. The model produced a mostly relevant Paper answer, although the checker identified **6 incorrectly placed items** from the Acceptable Items section. Importantly, **0 Paper reference answers were used as training targets**, confirming Paper was unseen during fine-tuning.

The result shows that the model can generalize to an unseen category when given its retrieved policy context, but the answer still requires the checker to ensure correct organization.


## **Step 19: Readability**

In [36]:
# Install textstat once if it is not already installed
# Install textstat
# !pip install textstat

# Import textstat for measuring readability
import textstat


# Split the answer into individual sentences
def split_sentences(text):

    # Use punctuation followed by a space to identify sentence boundaries
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


# Count pairs of sentences that are very similar to each other
def near_duplicate_pairs(text, threshold=0.5):

    # Convert each sentence into a set of lowercase words
    word_sets = [set(re.findall(r"[a-z]+", s.lower())) for s in split_sentences(text)]

    # Start the duplicate-pair count at zero
    pairs = 0

    # Compare each sentence with the other sentences
    for i in range(len(word_sets)):

        # Compare the current sentence with sentences after it
        for j in range(i + 1, len(word_sets)):

            # Find all unique words across the two sentences
            union = word_sets[i] | word_sets[j]

            # Check how many words the two sentences have in common
            if union and len(word_sets[i] & word_sets[j]) / len(union) >= threshold:

                # Count the pair if at least 50% of the words overlap
                pairs += 1

    # Return the number of similar sentence pairs
    return pairs


# Create an empty list to store readability results
rows = []

# Check the generated answer for each waste category
for category in CATEGORIES:

    # Get the generated answer for the category
    answer = outputs[category]

    # Split the answer into sentences
    sentences = split_sentences(answer)

    # Store word count, sentence count, readability, and repetition measures
    rows.append({"category": category, "words": len(answer.split()), "sentences": len(sentences),
                 "words per sentence": round(len(answer.split()) / len(sentences), 1),
                 "Flesch ease": round(textstat.flesch_reading_ease(answer), 1),
                 "grade level": round(textstat.flesch_kincaid_grade(answer), 1),
                 "near-duplicate pairs": near_duplicate_pairs(answer)})


# Convert the results into a DataFrame
readability = pd.DataFrame(rows)

# Display the readability results for each category
print(readability.to_string(index=False))

# Calculate the average readability measures across all categories
print(readability[["words per sentence", "Flesch ease", "grade level", "near-duplicate pairs"]].mean().round(2).to_dict())

           category  words  sentences  words per sentence  Flesch ease  grade level  near-duplicate pairs
          Cardboard    106          9                11.8         48.0          9.5                     0
      Food Organics     98          9                10.9         54.2          8.4                     0
              Glass     95         10                 9.5         40.5         10.0                     0
              Metal    110         10                11.0         40.3         10.4                     0
Miscellaneous Trash     99          9                11.0         11.1         14.4                     1
              Paper     96         10                 9.6         41.1          9.9                     0
            Plastic    103         11                 9.4         35.5         10.6                     0
      Textile Trash    104          9                11.6         43.0         10.1                     0
         Vegetation    101         10         

The generated answers averaged **10.54 words per sentence**, with a **Flesch reading ease of 39.86** and an estimated **grade level of 10.31**. Most categories had no near-duplicate sentences, with only Miscellaneous Trash showing one duplicate pair. Overall, the answers were reasonably structured but required a relatively advanced reading level.


## Summary

* **System:** category in, hybrid retrieval (TF-IDF weight 0.25 plus MiniLM) within the category's 6 chunks, fine-tuned `flan-t5-small` with greedy decoding, section-aware checker, and rebuild from the sources if the answer fails. The main function is `generate_instructions(category)`.

* **Evidence it uses the documents:** all 9 categories reproduced their reference answers exactly in the full pipeline test; the edited fact was used in **9 of 9 categories**; no invented items were produced when a section was removed; and the model trained without Paper still produced a mostly relevant Paper answer with **0.853 token F1**.

* **What went wrong:** when a section was missing, the model sometimes filled the missing section with misplaced text. The held-out Paper test also placed **6 Acceptable Items** under "Do not recycle:". Simple word overlap did not detect these errors, but the section-aware checker did.

* **Limitations:** the references are templated, so the results mainly demonstrate robustness rather than broad language-generation capability; Paper was only one held-out category and shares a layout with other categories; the retrieval evaluation uses a small hand-written question set; the checker does not cover every type of generated text; readability is partly inherited from the policy text; and the experiments were performed with small datasets and single CPU runs.
